# 05. LangGraph 에이전트 테스트

Qwen2.5-7B-Instruct를 기반으로, 5개 tool을 적용한 멀티스텝 에이전트.

## 워크플로우
```
START
  ↓
supervisor (LLM intent 분류)
  ↓ (병렬 실행)
sensor / anomaly / manual / history
  ↓
synthesizer (LLM 진단)
  ↓ (intent==workorder인 경우)
workorder (LLM → draft_workorder tool)
  ↓
END
```

## 테스트 시나리오
1. 단순 상태 점검 — '1호 압축기 어제 상태 점검해줘' -> sensor + anomaly만
2. 이상 진단 — '3호 압축기 압력이 이상한데 원인 찾아줘' -> 전체 진단 (manual/history 포함)
3. 작업지시서 생성 — '2호 공기 누설로 보이는데 작업지시서 만들어줘' -> 워크오더까지

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_metropt3, SensorDB
from src.models.anomaly import AEAnomalyModel
from src.models.llm import LLMConfig, load_hf_model, chat
from src.rag.retriever import load_retriever
from src.agent.tools import configure_tools, ToolDeps
from src.agent.graph import build_agent

ARTIFACTS = PROJECT_ROOT / 'models_artifacts'

## 1. 데이터 / 모델 / 검색기 / Tool 의존성 준비

In [2]:
df = load_metropt3(data_dir=PROJECT_ROOT / 'data' / 'metropt3', downsample='1min')
db = SensorDB(df=df)

ae = AEAnomalyModel.load(ARTIFACTS / 'convae_v1.pt', device='cpu')
anomaly_meta = json.loads((ARTIFACTS / 'anomaly_meta.json').read_text(encoding='utf-8'))
manual_r = load_retriever(ARTIFACTS / 'rag' / 'manual')
history_r = load_retriever(ARTIFACTS / 'rag' / 'history')

configure_tools(ToolDeps(
    sensor_db=db,
    anomaly_model=ae,
    anomaly_threshold=anomaly_meta['convae']['threshold'],
    anomaly_window=anomaly_meta['window'],
    anomaly_stride=anomaly_meta['stride'],
    manual_retriever=manual_r,
    history_retriever=history_r,
))
print('tools configured')

d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


tools configured


d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


## 2. Qwen2.5-7B-Instruct 로드 (INT4)

RTX 4060 8GB 기준 로드 가능

In [3]:
cfg = LLMConfig(model_id='Qwen/Qwen2.5-7B-Instruct', quantization='int4')
model, tokenizer = load_hf_model(cfg)
print(f'model loaded on {next(model.parameters()).device}')

def llm_chat(messages):
    return chat(messages, model, tokenizer, max_new_tokens=512, temperature=0.2)

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "d:\ProgramData\anaconda3\envs\transformer_learn\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "d:\ProgramData\anaconda3\envs\transformer_learn\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "d:\ProgramData\anaconda3\envs\transformer_learn\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\ProgramData\anaconda3\envs\transformer_learn\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

model loaded on cuda:0


## 3. 에이전트 빌드

In [4]:
agent = build_agent(llm_chat)
print('agent compiled')

agent compiled


In [5]:
def run_scenario(query: str, equipment_id: str | None = None, time_range=None):
    print(f'\n{"="*70}\nQUERY: {query}\n{"="*70}')
    init = {'query': query, 'equipment_id': equipment_id, 'time_range': time_range, 'trace': []}
    final = agent.invoke(init)

    print(f"\n[intent] {final.get('intent')}  [equipment] {final.get('equipment_id')}")
    print(f"[needs] sensor={final.get('needs_sensor')} anomaly={final.get('needs_anomaly')}"
           f" manual={final.get('needs_manual')} history={final.get('needs_history')}")

    print('\n--- TRACE ---')
    for t in final.get('trace', []):
        print(f"  [{t['ts']}] {t['node']:11s} {t['info']}")

    print('\n--- DIAGNOSIS ---')
    print(final.get('diagnosis'))

    if final.get('workorder'):
        print('\n--- WORKORDER ---')
        print(final['workorder'])
    return final

## 4. 시나리오 1 - 단순 상태 점검

정상 구간을 지정해 sensor 조회 + 이상 탐지가 NORMAL을 반환하는지 확인.

In [6]:
_ = run_scenario(
    query='1호 압축기 2월 15일 상태 점검해줘. 이상 없어 보이는지 확인 부탁해.',
    equipment_id='APU-01',
    time_range=('2020-02-15T00:00:00', '2020-02-17T00:00:00'),
)


QUERY: 1호 압축기 2월 15일 상태 점검해줘. 이상 없어 보이는지 확인 부탁해.

[intent] status_check  [equipment] APU-01
[needs] sensor=False anomaly=False manual=False history=False

--- TRACE ---
  [2026-05-05T15:28:13] supervisor  {'raw': '{\n  "intent": "status_check",\n  "equipment_id": "APU-01",\n  "needs_sensor": false,\n  "needs_anomaly": false,\n  "needs_manual": false,\n  "needs_history": false\n}', 'parsed': {'intent': 'status_check', 'equipment_id': 'APU-01', 'needs_sensor': False, 'needs_anomaly': False, 'needs_manual': False, 'needs_history': False}, 'elapsed': 4.008}
  [2026-05-05T15:28:26] synthesizer {'elapsed': 13.085}

--- DIAGNOSIS ---
1. **현재 상태 요약**: 1호 압축기의 최근 상태 점검 결과, 주요 센서 데이터는 정상 범위 내에 유지되고 있으며, 이상 징후는 관찰되지 않았습니다.

2. **진단 결론**: 현재 1호 압축기는 정상 작동 중이며, 고장 유형은 특정하지 않습니다.

3. **근거**: 이력 데이터와 현재 센서 값을 분석한 결과, 모든 주요 지표가 정상 범위 내에 머물고 있습니다. 특히, 전압, 회전 수, 온도 등 주요 센서의 측정값이 정상 범위 내에 위치하고 있습니다.

4. **권장 다음 조치**:
   - 정기적인 상태 점검을 지속합니다.
   - 장기적인 모니터링을 통해 장비의 변화를 관찰합니다.
   - 필요한 경우 추가적인 센서 데이터 수집을 진

## 5. 시나리오 2 - 이상 의심 진단 (Air Leak 구간)

In [7]:
_ = run_scenario(
    query='3호 압축기 압력이 이상해 보이는데, 무슨 문제인지 진단하고 어떻게 대응해야 하는지 매뉴얼/이력 참고해서 알려줘.',
    equipment_id='APU-03',
    time_range=('2020-06-11T00:00:00', '2020-06-13T00:00:00'),
)


QUERY: 3호 압축기 압력이 이상해 보이는데, 무슨 문제인지 진단하고 어떻게 대응해야 하는지 매뉴얼/이력 참고해서 알려줘.

[intent] diagnosis  [equipment] APU-03
[needs] sensor=True anomaly=True manual=True history=True

--- TRACE ---
  [2026-05-05T15:28:40] supervisor  {'raw': '{\n  "intent": "diagnosis",\n  "equipment_id": "APU-03",\n  "needs_sensor": true,\n  "needs_anomaly": true,\n  "needs_manual": true,\n  "needs_history": true\n}', 'parsed': {'intent': 'diagnosis', 'equipment_id': 'APU-03', 'needs_sensor': True, 'needs_anomaly': True, 'needs_manual': True, 'needs_history': True}, 'elapsed': 3.141}
  [2026-05-05T15:28:40] anomaly     {'equipment_id': 'APU-03'}
  [2026-05-05T15:28:40] history     {'query': '3호 압축기 압력이 이상해 보이는데, 무슨 문제인지 진단하고 어떻게 대응해야 하는지 매뉴얼/이력 참고해서 알려줘.', 'equipment_id': 'APU-03'}
  [2026-05-05T15:28:40] manual      {'query': '3호 압축기 압력이 이상해 보이는데, 무슨 문제인지 진단하고 어떻게 대응해야 하는지 매뉴얼/이력 참고해서 알려줘.'}
  [2026-05-05T15:28:40] sensor      {'equipment_id': 'APU-03', 'start': '2020-06-11T00:00:00', 'end': '2020-06-13T00:00:00'

## 6. 시나리오 3 —-

In [8]:
_ = run_scenario(
    query='2호 압축기 어제 누설 의심 알람 들어왔어. 진단하고 작업지시서 초안 만들어줘.',
    equipment_id='APU-02',
    time_range=('2020-05-12T00:00:00', '2020-05-13T00:00:00'),
)


QUERY: 2호 압축기 어제 누설 의심 알람 들어왔어. 진단하고 작업지시서 초안 만들어줘.

[intent] diagnosis  [equipment] APU-02
[needs] sensor=True anomaly=True manual=True history=True

--- TRACE ---
  [2026-05-05T15:29:13] supervisor  {'raw': '{\n  "intent": "diagnosis",\n  "equipment_id": "APU-02",\n  "needs_sensor": true,\n  "needs_anomaly": true,\n  "needs_manual": true,\n  "needs_history": true\n}', 'parsed': {'intent': 'diagnosis', 'equipment_id': 'APU-02', 'needs_sensor': True, 'needs_anomaly': True, 'needs_manual': True, 'needs_history': True}, 'elapsed': 3.387}
  [2026-05-05T15:29:13] anomaly     {'equipment_id': 'APU-02'}
  [2026-05-05T15:29:13] history     {'query': '2호 압축기 어제 누설 의심 알람 들어왔어. 진단하고 작업지시서 초안 만들어줘.', 'equipment_id': 'APU-02'}
  [2026-05-05T15:29:13] manual      {'query': '2호 압축기 어제 누설 의심 알람 들어왔어. 진단하고 작업지시서 초안 만들어줘.'}
  [2026-05-05T15:29:13] sensor      {'equipment_id': 'APU-02', 'start': '2020-05-12T00:00:00', 'end': '2020-05-13T00:00:00'}
  [2026-05-05T15:29:39] synthesizer {'elapsed': 26.158}